In [ ]:
#코랩에서 한글시각화설정
!pip install -q koreanize-matplotlib
import koreanize_matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 14.4 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [ ]:
import torch
print("gpu" if torch.cuda.is_available() else "cpu")
print('torch버전:',torch.__version__)

cpu
torch버전: 2.11.0+cpu


In [ ]:
# DNN을 활용한 심장병 발병예측(과거에는 텐서플로로 했음. 이제는 파이토치로 함)
1. 데이터 불러오기
- 결측치 처리(이미 완료)
- 원-핫 인코딩(y는 이미 완료. X도 쉽게 하는 방법도 있음)
- 원핫은, 변수가 카테코리화면인 경우임. 표준정규화는 연속형
- csv백업(구글드라이브 특정 폴더에 저장할 것임. 그냥 세이브하면 닫으며 다 날라감.)
2. feature(X)와 타겟(y)을 분리
3. 훈련/테스트 데이터 분할함 (==> 여기까지는 파이토치도 동일함)
4. Dataset클래스 정의해야 함. 그리고 데이터로더도 생성해야 함. 즉, 텐서플로우는 바로 모델 들어감. 파이토치는 데이터 설계작업을 들어가야 함
5. 모델정의도 클래스화 시킴. 파이토치, 텐서 등을 모두다 딥러닝. 머신러닝 다 해보는 것
6. 손실하마수와 옵티마이저를 세팅(ex. 렐루, 아담, 등등)
7. 학습루프를 돌림(파이토치는 내가 설정해줘야 함. 텐서에서는 fit만 하면 됨. 쉬움. 단, 텐서 1버전에서는 for문을 내가 돌림 이렇게 파이토치를 해야함. 불편)
8. 학습끝나면 평가(테스트데이터로 함). 32개씩 가져와서 해야 하는데 배치싸이즈 32할거면 트레인, 테스트 모두다 해야 함
9. 예측

SyntaxError: invalid syntax (3077170594.py, line 2)

In [ ]:
#1. 데이터불러오기

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn #Sequential(), Dropout(), Linear()
import torch.optim as optim

In [ ]:
# 심장병 데이터셋 파일 불러오기
# https://github.com/4aix/data/blob/master/heart.csv =>
# https://raw.githubusercontent.com/4aix/data/master/heart.csv
url = 'https://raw.githubusercontent.com/4aix/data/master/heart.csv'
df = pd.read_csv(url)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


# 2. 데이터전처리
- 카테코리칼 변수는 원핫인코딩
- 연속형 수치 변수는 정규화(MinMaxScaler)..내 마음이다.
총 14개 변수 중 categorical 변수는 4개이다

In [ ]:
# feature변수의 원핫인코딩
from pandas.core.arrays import categorical
# ['cp', 'restecg', 'slope', 'thal' ] 카테고리컬한 변수 컬럼이름
categorical_cols = ['cp', 'restecg', 'slope', 'thal' ]
# 여러 열을 동시에 원핫인코딩
df = pd.get_dummies(df, columns=categorical_cols)
df.shape

(303, 24)

In [ ]:
# feature의 scale 조정
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
minmax_scaler = MinMaxScaler()
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
# df = df.copy()
df[numeric_cols] = minmax_scaler.fit_transform(df[numeric_cols])
df.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,target,...,restecg_0,restecg_1,restecg_2,slope_0,slope_1,slope_2,thal_0,thal_1,thal_2,thal_3
0,0.708333,1,0.481132,0.244292,1,0.603053,0,0.370968,0.0,1,...,True,False,False,True,False,False,False,True,False,False
1,0.166667,1,0.339623,0.283105,0,0.885496,0,0.564516,0.0,1,...,False,True,False,True,False,False,False,False,True,False
2,0.250000,0,0.339623,0.178082,0,0.770992,0,0.225806,0.0,1,...,True,False,False,False,False,True,False,False,True,False
3,0.562500,1,0.245283,0.251142,0,0.816794,0,0.129032,0.0,1,...,False,True,False,False,False,True,False,False,True,False
4,0.583333,0,0.245283,0.520548,0,0.702290,1,0.096774,0.0,1,...,False,True,False,False,False,True,False,False,True,False


In [ ]:
df.to_csv('heart_preprocessed.csv',index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
